# 176 — Aprendizaje continuo y adaptación

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución 1 — EWC a mano

```text
L(θ) = (θ₁−3)² + θ₂² + (2/2)·[4(θ₁−1)² + 0·(θ₂−2)²]

∂L/∂θ₁ = 2(θ₁−3) + 8(θ₁−1) = 0 → 10θ₁ = 14 → θ₁ = 1.4
∂L/∂θ₂ = 2θ₂ = 0                             → θ₂ = 0.0
```

c) El peso 1 quedó anclado: su Fisher (4.0) indica que la tarea A depende de él, así
que B solo lo mueve de 1.0 a 1.4 (no al 3.0 que preferiría). El peso 2 tiene `F=0`:
A no lo usa, así que va exactamente a lo que pide B (0.0) aunque estaba en 2.0.


In [ ]:
# Verificación numérica por descenso de gradiente
theta = [0.0, 0.0]
lam, F, tA = 2.0, (4.0, 0.0), (1.0, 2.0)
for _ in range(5000):
    g1 = 2 * (theta[0] - 3) + lam * F[0] * (theta[0] - tA[0])
    g2 = 2 * (theta[1] - 0) + lam * F[1] * (theta[1] - tA[1])
    theta = [theta[0] - 0.01 * g1, theta[1] - 0.01 * g2]
print(round(theta[0], 3), round(theta[1], 3))  # 1.4, 0.0
assert abs(theta[0] - 1.4) < 1e-3 and abs(theta[1]) < 1e-3


## Solución 2 — Olvido y BWT

a) Olvido de T1 = max(0.95, 0.60, 0.35) − 0.35 = **0.60**.
   Olvido de T2 = 0.93 − 0.55 = **0.38**.

b) Media final = (0.35 + 0.55 + 0.94)/3 = **0.613**.
   BWT = media de `R[3][j] − R[j][j]` para j<3 = ((0.35−0.95) + (0.55−0.93))/2 =
   (−0.60 − 0.38)/2 = **−0.49** (transferencia hacia atrás fuertemente negativa).

c) La diagonal es alta (~0.93-0.95: cada tarea se aprende bien al momento) pero la
última fila se hunde para tareas viejas: aprende siempre, retiene poco. Además el
olvido es mayor cuanto más antigua la tarea (T1 sufre más que T2).


In [ ]:
R = {1: {1: 0.95}, 2: {1: 0.60, 2: 0.93}, 3: {1: 0.35, 2: 0.55, 3: 0.94}}
olvido_T1 = max(R[i][1] for i in R if 1 in R[i]) - R[3][1]
olvido_T2 = max(R[i][2] for i in R if 2 in R[i]) - R[3][2]
media_final = sum(R[3].values()) / 3
bwt = ((R[3][1] - R[1][1]) + (R[3][2] - R[2][2])) / 2
print(olvido_T1, olvido_T2, round(media_final, 3), round(bwt, 3))
assert abs(bwt + 0.49) < 1e-9


## Solución 3 — Laboratorio y semilla

a) Permanecen idénticas las claves estructurales del contrato (`kind`, la forma de
`evidence` y `limitations`); cambian los valores derivados de la semilla registrada
en la configuración. El contrato es estable; el contenido, reproducible por semilla.

b) La limitación tipo "demo educativa / sin persistencia entre ejecuciones" es el
equivalente directo: cada `run_lab` parte de cero, no hay estado consolidado — es un
sistema con plasticidad total y estabilidad nula, el extremo opuesto a un modelo
congelado.


In [ ]:
r1 = run_lab("frontier", seed=176)
r2 = run_lab("frontier", seed=174)
assert r1["kind"] == r2["kind"] == "frontier"
assert set(r1.keys()) == set(r2.keys())
print("claves comunes:", sorted(r1.keys()))
print("limitations:", r1["limitations"])


## Solución 4 — Método por escenario

a) **Regularización (EWC o similar)**: no puede retener datos crudos fuera de origen,
así que el replay clásico queda descartado; EWC solo conserva estadísticos de pesos.

b) **Arquitectura (LoRA/adaptador por cliente)**: la identidad de tarea se conoce en
inferencia, 40 módulos pequeños son baratos y el olvido es nulo por construcción.

c) **Replay con buffer comprimido**: sin etiqueta de tarea en inferencia la familia
arquitectural se complica, y con 200 tareas la aproximación de EWC se degrada; un
buffer pequeño y bien muestreado es lo más robusto dentro del presupuesto de memoria.
